##### Creating tables in SQL

In [0]:
%sql
USE CATALOG test;
USE SCHEMA sql;

DROP TABLE IF EXISTS population;
CREATE TABLE population (
    country VARCHAR(50),
    continent VARCHAR(10),
    total_population BIGINT
);

INSERT INTO population (country, continent, total_population)
VALUES
('India', 'Asia', 1428627663),
('China', 'Asia', 1425671352),
('United States', 'Americas', 339996564),
('Indonesia', 'Asia', 277534123),
('Pakistan', 'Asia', 240485658),
('Nigeria', 'Africa', 223804632),
('Brazil', 'Americas', 216422446),
('Bangladesh', 'Asia', 172954319),
('Russia', 'Europe', 144444359),
('Mexico', 'Americas', 128455567);

num_affected_rows,num_inserted_rows
10,10


##### Creating dataframe & then saving as table in PYSPARK

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("country", StringType(), True),
    StructField("continent", StringType(), True),
    StructField("total_population", LongType(), True)
])
data = [
    ("India", "Asia", 1428627663),
    ("China", "Asia", 1425671352),
    ("United States", "Americas", 339996564),
    ("Indonesia", "Asia", 277534123),
    ("Pakistan", "Asia", 240485658),
    ("Nigeria", "Africa", 223804632),
    ("Brazil", "Americas", 216422446),
    ("Bangladesh", "Asia", 172954319),
    ("Russia", "Europe", 144444359),
    ("Mexico", "Americas", 128455567)
]
population_df = spark.createDataFrame(data, schema)
population_df.write.mode("overwrite").saveAsTable("test.pyspark.population")

population_df = spark.read.table("test.pyspark.population")
population_df.display()

country,continent,total_population
India,Asia,1428627663
China,Asia,1425671352
United States,Americas,339996564
Indonesia,Asia,277534123
Pakistan,Asia,240485658
Nigeria,Africa,223804632
Brazil,Americas,216422446
Bangladesh,Asia,172954319
Russia,Europe,144444359
Mexico,Americas,128455567


##### - Find the countries with a population higher than the average population of the top 10 populated countries
##### - Find the population of countries, which are the only countries for their continent amongst the top 10 most populated countries.

###### Solving above with **SQL**

In [0]:
%sql
-- Find the countries with a population higher than the average population of the top 10 populated countries
select * from population where total_population > (select avg(total_population) from population);
select * from population where total_population = (select max(total_population) from population);

-- Find the population of countries, which are the only countries for their continent amongst the top 10 most populated countries.
select * from population where continent in (select continent from population group by continent having count(country) = 1);

country,continent,total_population
Nigeria,Africa,223804632
Russia,Europe,144444359


###### Solving using **PYSPARK**

In [0]:
from pyspark.sql.functions import avg, max, col, count

avg_pop = population_df.select(
    avg("total_population")
).collect()[0][0]

max_pop = population_df.select(
    max("total_population")
).collect()[0][0]

population_df.filter(
    col("total_population") > avg_pop
).show()

population_df.filter(
    col("total_population") == max_pop
).show()


# Find the population of countries, which are the only countries for their continent amongst the top 10 most populated countries.
a = population_df.groupBy(
    col("continent").alias("con")
).agg(
    count("country").alias("count_country"),
).filter(
    col("count_country") == 1
)

b = population_df.alias("b")
b.join(
    a,
    a.con == b.continent
).select(
    b.country,
    b.continent,
    b.total_population 
).display()

+-------+---------+----------------+
|country|continent|total_population|
+-------+---------+----------------+
|  India|     Asia|      1428627663|
|  China|     Asia|      1425671352|
+-------+---------+----------------+

+-------+---------+----------------+
|country|continent|total_population|
+-------+---------+----------------+
|  India|     Asia|      1428627663|
+-------+---------+----------------+



country,continent,total_population
Nigeria,Africa,223804632
Russia,Europe,144444359


- #### Get population information of countries with highest population in each continent

##### Solving using SQL

In [0]:
%sql
-- Get population information of countries with highest population in each continent
-- Normal legacy method
select * from population where (continent, total_population) in (select continent, max(total_population) from population group by continent);

-- Window function
select continent, country, total_population from (
    select
        *,
        row_number() over(
            partition by continent
            order by total_population desc
        ) as rn
    from population
) t
where rn = 1;

continent,country,total_population
Africa,Nigeria,223804632
Americas,United States,339996564
Asia,India,1428627663
Europe,Russia,144444359


##### Solving using PYSPARK

In [0]:
# Get population information of countries with highest population in each continent

# Using legacy method
from pyspark.sql.functions import avg, max, col, count, dense_rank
a = population_df \
        .groupBy(col("continent").alias("con")) \
        .agg(max("total_population").alias("max"))

b = population_df.alias("b")
b.join(
    a,
    (a.con == b.continent) & (a.max == b.total_population)
).select(
    b.country,
    b.continent,
    b.total_population 
).display()


## Using window function
from pyspark.sql.window import Window
w = Window.partitionBy(col("continent")).orderBy(col("total_population").desc())
result = (
    population_df \
    .withColumn("rank", dense_rank().over(w)) \
    .filter(col("rank") == 1).drop(col("rank")).orderBy(col("total_population").desc())
)
result.display()

country,continent,total_population
India,Asia,1428627663
United States,Americas,339996564
Nigeria,Africa,223804632
Russia,Europe,144444359


country,continent,total_population
India,Asia,1428627663
United States,Americas,339996564
Nigeria,Africa,223804632
Russia,Europe,144444359


- ##### Add an additional column to the result set of population table to include the total population of the continent

##### Solving using SQL

In [0]:
%sql
-- Add an additional column to the result set of population table to include the total population of the continent

-- Using normal Subquery
select p.continent, p.country, p.total_population, a.continent_population from population as p
join (select continent, sum(total_population) as continent_population from population group by continent) as a 
on p.continent = a.continent;

-- Using Window function
select 
    continent,
    country,
    total_population,
    sum(total_population) over (
        partition by continent
    ) as continent_population
from population;

continent,country,total_population,continent_population
Asia,India,1428627663,3545273115
Asia,China,1425671352,3545273115
Americas,United States,339996564,684874577
Asia,Indonesia,277534123,3545273115
Asia,Pakistan,240485658,3545273115
Africa,Nigeria,223804632,223804632
Americas,Brazil,216422446,684874577
Asia,Bangladesh,172954319,3545273115
Europe,Russia,144444359,144444359
Americas,Mexico,128455567,684874577


##### Using PYSPARK

In [0]:
# Add an additional column to the result set of population table to include the total population of the continent

from pyspark.sql.window import Window
from pyspark.sql.functions import col, sum

w = Window.partitionBy(col("continent"))
population_df.withColumn(
    "continent_population",
    sum(col("total_population")).over(w)
).display()

country,continent,total_population,continent_population
India,Asia,1428627663,3545273115
China,Asia,1425671352,3545273115
United States,Americas,339996564,684874577
Indonesia,Asia,277534123,3545273115
Pakistan,Asia,240485658,3545273115
Nigeria,Africa,223804632,223804632
Brazil,Americas,216422446,684874577
Bangladesh,Asia,172954319,3545273115
Russia,Europe,144444359,144444359
Mexico,Americas,128455567,684874577


#### Corelated Subquery

##### Find the highest populated country in each continent

In [0]:
%sql

-- Using regular Subquery
select * from population where (continent, total_population) in (select continent, max(total_population) from population group by continent);

-- Using Windows Function
select country, continent, total_population from 
(
    select 
        *,
        dense_rank() over(
            partition by continent
            order by total_population desc
        ) as rank
    from population
) as t
where rank = 1;

-- Using corelated subquery
select 
    a.country, 
    a.continent, 
    a.total_population 
from population a 
where a.total_population = (
        select max(b.total_population)
        from population b
        where a.continent = b.continent
    );


country,continent,total_population
India,Asia,1428627663
United States,Americas,339996564
Nigeria,Africa,223804632
Russia,Europe,144444359


-  ##### Find all the countries with the population that is greater than or equal to the average population countries in that continent.

###### SQL

In [0]:
%sql

-- 1. Uisng corelated subquery
select 
    country, 
    continent, 
    total_population 
from population a
where 
    total_population >= (
    select avg(total_population) from population b 
    where a.continent = b.continent
);


-- 2. Using regular subquery + join
select 
    a.country, 
    a.continent, 
    a.total_population 
from population a 
join (
    select 
        continent, 
        avg(total_population) as avg
    from population group by continent
    ) as b
on a.continent = b.continent
where a.total_population >= b.avg;


-- 3. Using Windows function
select country, continent, total_population from 
(
    select 
        *,
        avg(total_population) over (
            partition by continent
        ) as avg
    from population
) as t
where total_population >= avg;


country,continent,total_population
India,Asia,1428627663
China,Asia,1425671352
United States,Americas,339996564
Nigeria,Africa,223804632
Russia,Europe,144444359


###### PYSPARK

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, avg

w = Window.partitionBy(col("continent"))
result = population_df.withColumn(
    "avg_pop",
    avg(col("total_population")).over(w)
).filter(
    col("total_population") >= col("avg_pop")
).select(
    col("country"),
    col("continent"),
    col("total_population")
)

result.display()

country,continent,total_population
India,Asia,1428627663
China,Asia,1425671352
United States,Americas,339996564
Nigeria,Africa,223804632
Russia,Europe,144444359
